
# **Step 1:** The Ingestion Block (Connecting & Extracting)

This block sets up a secure connection socket to your locally running MetaTrader 5 terminal, requests a clean slice of historical time-series data, and converts the raw binary structure into a localized Pandas DataFrame.
Create a new cell and write the following code:

In [ ]:
import MetaTrader5 as mt5 
import pandas as pd 
import numpy as np

def connect_and_fetch_forex_data(symbol:str, timeframe,num_bars:int) -> pd.DataFrame:
    if not mt5. initialize():
        raise RuntimeError(f"Could not Connect to MT5. \n Error: {mt5.last_error()}")

    print("Initialized Connection to MT5 Terminal.....")

    mt5.symbol_select(symbol,True)

    raw_rates = mt5.copy_rates_from_pos(symbol,timeframe,0,num_bars)

    mt5.shutdown()
    print("MT5 Terminal Closed Successfully!")

    df = pd.DataFrame(raw_rates)

    df['time'] = pd.to_datetime(df['time'],unit='s')
    df.set_index('time',inplace=True)

    essential_columns = ['open', 'high', 'low', 'close', 'tick_volume']
    return df[essential_columns]

aw_eurusd_df =  connect_and_fetch_forex_data("EURUSD",mt5.TIMEFRAME_H1,5000)
print(f"Matrix Shape Extracted: {aw_eurusd_df.shape}")

Initialized Connection to MT5 Terminal.....
MT5 Terminal Closed Successfully!
Matrix Shape Extracted: (5000, 5)


# **Step 2 :** Feature Enginering, here I am gonna take the raw rates and then create the Engineered stationary features.

In [ ]:
def engineer_stationary_features(raw_df: pd.DataFrame) -> pd.DataFrame: 
    matrix = raw_df.copy()

    matrix['log_return'] = np.log(matrix['close']/matrix['close'].shift(1))

    matrix['high_low_span'] = matrix['high'] - matrix['low']
    matrix['volatility_atr'] = matrix['high_low_span'].rolling(window=14).mean()

    matrix['raw_hour'] = matrix.index.hour
    
    matrix['hour_sin'] = np.sin(2 * np.pi * matrix['raw_hour'] / 24.0)
    matrix['hour_cos'] = np.cos(2 * np.pi * matrix['raw_hour'] / 24.0)

    matrix.dropna(inplace=True)

    final_features = matrix.drop(columns=['open','high','low','raw_hour','high_low_span'])

    return final_features 

final_feature_matrix = engineer_stationary_features(raw_eurusd_df)
print("Pipeline complete! Displaying top 3 rows of processed ML input matrix:")
print(final_feature_matrix[['close', 'log_return', 'volatility_atr', 'hour_sin', 'hour_cos']].head(3))

 # This function contains the loop that steps through your historical rows, projects the dynamic ATR brackets forward in time, and extracts the target ground-truth label.

In [ ]:
def generate_triple_barrier_labels(df: pd.DataFrame, pt_mult: float = 2.0, sl_mult: float = 1.0, max_bars: int = 24) -> pd.Series:

    labels = np.zeros(len(df))
    
    close_prices = df['close'].values
    atr_values = df['volatility_atr'].values
    
    total_rows = len(df)
    
    for t in range(total_rows - max_bars):
        current_close = close_prices[t]
        current_atr = atr_values[t]
        
        upper_barrier = current_close + (current_atr * pt_mult)
        lower_barrier = current_close - (current_atr * sl_mult)
        
        forward_path = close_prices[t + 1 : t + 1 + max_bars]
        
        profit_crossings = np.where(forward_path >= upper_barrier)[0]
        loss_crossings = np.where(forward_path <= lower_barrier)[0]
        
        # Find the absolute first index where each barrier was breached
        first_profit_idx = profit_crossings[0] if len(profit_crossings) > 0 else None
        first_loss_idx = loss_crossings[0] if len(loss_crossings) > 0 else None
        
        # 5. Evaluate which barrier was hit chronologically first
        if first_profit_idx is not None and first_loss_idx is not None:
            if first_profit_idx < first_loss_idx:
                labels[t] = 1   # Profit hit first
            else:
                labels[t] = -1  # Loss hit first
        elif first_profit_idx is not None:
            labels[t] = 1       # Only profit barrier was hit
        elif first_loss_idx is not None:
            labels[t] = -1      # Only loss barrier was hit
        else:
            labels[t] = 0       # Neither barrier hit within 24 bars: Time Expiry
            
    # Convert our results array into a structured Pandas Series aligned with our time index
    return pd.Series(labels, index=df.index, name='target')

target_labels = generate_triple_barrier_labels(final_feature_matrix, pt_mult=2.0, sl_mult=1.0, max_bars=24)

final_feature_matrix['target'] = target_labels
clean_ml_matrix = final_feature_matrix.iloc[:-24].copy()

print(f"Labeling complete! Processed Dataset Shape: {clean_ml_matrix.shape}")


In [ ]:
class_counts = clean_ml_matrix['target'].value_counts()
print("--- TARGET DISTRIBUTION BALANCE ---")
print(f"Take Profit (+1.0): {class_counts.get(1.0, 0)} trades")
print(f"Stop Loss (-1.0):   {class_counts.get(-1.0, 0)} trades")
print(f"Time Expiry (0.0):  {class_counts.get(0.0, 0)} trades\n")

class_percentages = clean_ml_matrix['target'].value_counts(normalize=True) * 100
print("--- PERCENTAGE PROFILE ---")
print(f"Take Profit: {class_percentages.get(1.0, 0):.2f}%")
print(f"Stop Loss:   {class_percentages.get(-1.0, 0):.2f}%")
print(f"Time Expiry: {class_percentages.get(0.0, 0):.2f}%")

assert class_counts.get(0.0, 0) < (len(clean_ml_matrix) * 0.85), "Warning: Over 85% of your data is marked as Time Expiry. Your ATR barriers may be too wide!"

assert set(clean_ml_matrix['target'].unique()).issubset({-1.0, 0.0, 1.0}), "Error: Invalid classes discovered in target array!"

print("Sprint 2 Data Target Assembly complete! Ready for Purged Cross-Validation.")

